# SpikedLM — longer Shakespeare run

Train the JAX **attention + Spiking LSTM** LM (byte-BPE + Nord-style spike-rate / LIF knobs) longer than the smoke config.

| Config | Steps | Size | Goal |
|--------|------:|------|------|
| `llm_smoke` | 200 | ~108k | pipeline check |
| `llm_toy` | 5000 | 4×128 | first readable run |
| **`llm_large` (this notebook)** | **15000** | **6×256** | stronger Shakespeare-ish text |

**Local:** project `.venv` kernel → Run All.

**Colab (GPU):** Runtime → GPU → Run All.

> **Colab:** setup always wipe+reclones `dev/other` (`FORCE_RECLONE=True`). Restart session if cwd is broken, then Run All.

Expect wall time on CPU: roughly **1–3+ hours**. GPU is much faster.


## 1. Environment and repo root

In [1]:
from __future__ import annotations

import os
import shutil
import subprocess
import sys
from pathlib import Path

IN_COLAB = Path("/content").exists()
REPO_URL = "https://github.com/AlexWoods1/Spiking-Neural-Network.git"
REPO_REF = "dev/other"
# * Colab checkouts get dirty/broken easily — always wipe + reclone by default.
FORCE_RECLONE = True


def _has_llm_spiked(root: Path) -> bool:
    return (root / "src" / "spiking_neural_network" / "LLM_spiked" / "model.py").is_file()


def _run(cmd: list[str], cwd: Path | None = None) -> None:
    print("+", " ".join(cmd))
    subprocess.run(cmd, cwd=str(cwd) if cwd else None, check=True)


if IN_COLAB:
    # * Never stay inside a deleted tree — reset cwd first.
    os.chdir("/content")
    ROOT = Path("/content/Spiking-Neural-Network")
    if FORCE_RECLONE or not (ROOT / "pyproject.toml").is_file() or not _has_llm_spiked(ROOT):
        if ROOT.exists():
            print("Removing checkout:", ROOT)
            shutil.rmtree(ROOT, ignore_errors=True)
        _run(
            [
                "git",
                "clone",
                "--branch",
                REPO_REF,
                "--single-branch",
                REPO_URL,
                str(ROOT),
            ],
            cwd=Path("/content"),
        )
    os.chdir(ROOT)
    print(subprocess.check_output(["git", "log", "-1", "--oneline"], text=True).strip())

    pyproject = ROOT / "pyproject.toml"
    if not pyproject.is_file():
        raise FileNotFoundError(
            f"Clone failed — missing {pyproject}. Runtime → Restart session, then re-run."
        )
    text = pyproject.read_text(encoding="utf-8")
    if 'requires-python = ">=3.14"' in text:
        pyproject.write_text(
            text.replace('requires-python = ">=3.14"', 'requires-python = ">=3.11"'),
            encoding="utf-8",
        )
        print("Patched requires-python to >=3.11")

    _run([sys.executable, "-m", "pip", "install", "-q", "optax", "pyyaml", "numpy", "tqdm"])
    try:
        import jax as _jax_probe

        _devs = [str(d).lower() for d in _jax_probe.devices()]
        if not any("cuda" in d or "gpu" in d for d in _devs):
            _run([sys.executable, "-m", "pip", "install", "-q", "-U", "jax[cuda12]"])
    except Exception:
        _run([sys.executable, "-m", "pip", "install", "-q", "-U", "jax[cuda12]"])
else:
    ROOT = Path.cwd()
    if not (ROOT / "src" / "spiking_neural_network").is_dir():
        for candidate in [ROOT, *ROOT.parents]:
            if (candidate / "src" / "spiking_neural_network").is_dir():
                ROOT = candidate
                break
    os.chdir(ROOT)

src = str(ROOT / "src")
sys.path = [p for p in sys.path if "spiking_neural_network" not in p.replace("\\", "/")]
if src not in sys.path:
    sys.path.insert(0, src)

import importlib
import jax

print("ROOT", ROOT)
print("JAX", jax.__version__, "devices", jax.devices())
print("LLM_spiked present:", _has_llm_spiked(ROOT))
if not _has_llm_spiked(ROOT):
    raise SystemExit("LLM_spiked missing after clone.")
importlib.invalidate_caches()
from spiking_neural_network.LLM_spiked.data import CharTokenizer  # noqa: F401

print("Import OK: spiking_neural_network.LLM_spiked")


Removing checkout: /content/Spiking-Neural-Network
+ git clone --branch dev/other --single-branch https://github.com/AlexWoods1/Spiking-Neural-Network.git /content/Spiking-Neural-Network
52427fd Fix jax.jit donate_argnums decorator for older Colab JAX.
Patched requires-python to >=3.11
+ /usr/bin/python3 -m pip install -q optax pyyaml numpy tqdm
ROOT /content/Spiking-Neural-Network
JAX 0.7.2 devices [CudaDevice(id=0)]
LLM_spiked present: True
Import OK: spiking_neural_network.LLM_spiked


## 1b. Colab only — upload `LLM_spiked` if GitHub is missing it

On your PC (repo root), create a zip:

```powershell
Compress-Archive -Path src\spiking_neural_network\LLM_spiked,configs,scripts\prepare_shakespeare.py,scripts\train_llm.py -DestinationPath llm_spiked_bundle.zip -Force
```

Then run the next cell and select `llm_spiked_bundle.zip`. Skip this section if setup already printed `Import OK`.

In [12]:
import io
import shutil
import zipfile
from pathlib import Path

assert "ROOT" in globals(), "Run the setup cell first."

if _has_llm_spiked(ROOT):
    print("LLM_spiked already present — skip upload.")
elif not IN_COLAB:
    raise SystemExit(
        "LLM_spiked missing locally. Build/open this repo on the machine that has the package."
    )
else:
    from google.colab import files

    print("Upload llm_spiked_bundle.zip …")
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit("No file uploaded.")
    name, raw = next(iter(uploaded.items()))
    zpath = ROOT / name
    zpath.write_bytes(raw)
    with zipfile.ZipFile(io.BytesIO(raw)) as zf:
        zf.extractall(ROOT / "_bundle_extract")
    extracted = ROOT / "_bundle_extract"

    # * Accept either a nested LLM_spiked/ or src/spiking_neural_network/LLM_spiked/.
    candidates = list(extracted.rglob("LLM_spiked"))
    pkg = next((p for p in candidates if (p / "model.py").is_file()), None)
    if pkg is None:
        raise SystemExit(f"Could not find LLM_spiked/model.py inside {name}")
    dest = ROOT / "src" / "spiking_neural_network" / "LLM_spiked"
    if dest.exists():
        shutil.rmtree(dest)
    shutil.copytree(pkg, dest)

    # Optional extras from the same zip
    for rel in ("configs", "scripts"):
        src_extra = next((p for p in extracted.rglob(rel) if p.is_dir()), None)
        if src_extra is not None:
            for item in src_extra.iterdir():
                target = ROOT / rel / item.name
                target.parent.mkdir(parents=True, exist_ok=True)
                if item.is_file():
                    shutil.copy2(item, target)

    shutil.rmtree(extracted, ignore_errors=True)
    import importlib
    importlib.invalidate_caches()
    from spiking_neural_network.LLM_spiked.data import CharTokenizer  # noqa: F401
    print("Import OK after upload:", dest)


Upload llm_spiked_bundle.zip …


KeyboardInterrupt: 

## 2. Large-run config (`llm_large`)

Writes `configs/llm_large.yaml`. On OOM, set `BATCH_SIZE = 16`.


In [2]:
# --- knobs (edit these) ---
MAX_STEPS = 15000
BATCH_SIZE = 24          # drop to 16 on OOM
N_LAYER, N_HEAD, N_EMBD = 6, 8, 256
BLOCK_SIZE = 256
VOCAB_SIZE = 1024        # byte-BPE; overridden from tokenizer at train time
SAMPLE_INTERVAL = 1000   # mid-train samples are expensive
EVAL_INTERVAL = 250
CHECKPOINT_INTERVAL = 1000

CFG_PATH = ROOT / "configs" / "llm_large.yaml"
CFG_PATH.parent.mkdir(parents=True, exist_ok=True)
CFG_PATH.write_text(
    f"""# Generated by notebooks/train_llm_long.ipynb (llm_large)
model:
  n_layer: {N_LAYER}
  n_head: {N_HEAD}
  n_embd: {N_EMBD}
  block_size: {BLOCK_SIZE}
  vocab_size: {VOCAB_SIZE}
  dropout: 0.1
  bias: true
  v_th: 0.5
  leak: 0.5
  v_minus: -1.0
  v_plus: 2.0
  alpha: 1.0
  beta: 1.0
  learnable_lif: true
  leaky_clamp_slope: 0.2
  spike_rate_weight: 0.25
  target_rate_i: 0.3
  target_rate_f: 0.5
  target_rate_o: 0.4
  rate_floor: 0.05

train:
  batch_size: {BATCH_SIZE}
  max_steps: {MAX_STEPS}
  learning_rate: 3.0e-4
  weight_decay: 0.1
  beta1: 0.9
  beta2: 0.99
  warmup_steps: 300
  grad_clip: 1.0
  eval_interval: {EVAL_INTERVAL}
  eval_batches: 8
  sample_interval: {SAMPLE_INTERVAL}
  checkpoint_interval: {CHECKPOINT_INTERVAL}
  seed: 1337

data:
  dataset: shakespeare
  data_dir: data/shakespeare
  train_frac: 0.9

paths:
  out_dir: checkpoints/llm_large
  tokenizer_path: data/shakespeare/tokenizer.json
""",
    encoding="utf-8",
)
print("Wrote", CFG_PATH)


Wrote /content/Spiking-Neural-Network/configs/llm_large.yaml


## 3. Prepare Shakespeare + byte-BPE tokenizer (vocab 1024)

If you already have an old **char** `tokenizer.json` (vocab ~65), delete `data/shakespeare/tokenizer.json` and the `.bin` caches so this cell rebuilds BPE.

In [3]:
import importlib.util
import json

from spiking_neural_network.LLM_spiked.data import load_tokenizer
from spiking_neural_network.LLM_spiked.tokenizer import ByteBPETokenizer

# * Load prepare helpers without requiring scripts/ to be a package.
_prep_path = ROOT / "scripts" / "prepare_shakespeare.py"
_spec = importlib.util.spec_from_file_location("prepare_shakespeare", _prep_path)
_prep = importlib.util.module_from_spec(_spec)
assert _spec.loader is not None
_spec.loader.exec_module(_prep)

data_dir = ROOT / "data" / "shakespeare"
tok_path = data_dir / "tokenizer.json"
need_prepare = not (data_dir / "train.txt").is_file() or not tok_path.is_file()
if tok_path.is_file():
    raw = json.loads(tok_path.read_text(encoding="utf-8"))
    # * Rebuild if an old char tokenizer is still on disk.
    if "merges" not in raw:
        need_prepare = True

if need_prepare:
    text = _prep.download_shakespeare()
    train, val = _prep.split_train_val(text, 0.9)
    _prep.write_splits(data_dir, train, val)
    tok = ByteBPETokenizer()
    tok.train(text, VOCAB_SIZE)
    tok.save(tok_path)
    for bin_path in (data_dir / "train.bin", data_dir / "val.bin"):
        if bin_path.is_file():
            bin_path.unlink()
    print(f"Tokenizer vocab_size={tok.vocab_size} (byte-BPE)")
else:
    tok = load_tokenizer(tok_path)
    print(f"Reusing data in {data_dir} (vocab_size={tok.vocab_size})")


Wrote /content/Spiking-Neural-Network/data/shakespeare/train.txt (1,003,835 chars)
Wrote /content/Spiking-Neural-Network/data/shakespeare/val.txt (111,559 chars)
Tokenizer vocab_size=65


## 4. Train

Checkpoints land in `checkpoints/llm_large/ckpt_{step}_weights.pkl`.
Healthy progress: val CE from ~`ln(65)≈4.17` toward **~1.5 or lower** by ~10k–15k steps.


In [4]:
from spiking_neural_network.LLM_spiked.train import train

params = train(CFG_PATH)
print("Training finished. Final param tree keys:", list(params.keys()))

jax backend=gpu devices=[CudaDevice(id=0)]
matmul precision=bfloat16
Wrote data/shakespeare/train.bin (1,003,835 tokens)
Wrote data/shakespeare/val.bin (111,559 tokens)
params=4,819,712 vocab=65


train:   2%|▏         | 250/15000 [01:14<5:52:49,  1.44s/it, loss=2.31, lr=0.00025]

step 250: train 2.3338 val 2.3288


train:   3%|▎         | 500/15000 [02:06<1:53:04,  2.14it/s, loss=1.87, lr=0.0003]  

step 500: train 1.8740 val 1.9510


...

step 14750: train 1.3455 val 1.5815


train: 100%|█████████▉| 14999/15000 [52:11<00:00,  4.94it/s, loss=1.42, lr=3e-5]   

step 15000: train 1.4276 val 1.6484


train: 100%|██████████| 15000/15000 [52:12<00:00,  4.79it/s, loss=1.42, lr=3e-5]

--- sample @ 15000 ---

KRRRlMOMPPLLMEETTCCllUTTRRRRlTTYPRTTPeIS
---------------
Wrote checkpoints/llm_large/ckpt_15000_weights.pkl
Wrote checkpoints/llm_large/ckpt_15000.pkl
Training finished. Final param tree keys: ['blocks', 'ln_f', 'wpe', 'wte']


## 5. Generate from the last checkpoint

In [5]:
from spiking_neural_network.LLM_spiked.generate import generate, load_checkpoint

ckpt_dir = ROOT / "checkpoints" / "llm_large"
ckpts = sorted(ckpt_dir.glob("ckpt_*_weights.pkl"), key=lambda p: int(p.stem.split("_")[1]))
assert ckpts, f"No checkpoints in {ckpt_dir}"
ckpt = ckpts[-1]
print("Using", ckpt)

params, model_cfg, tok = load_checkpoint(ckpt)
text = generate(
    params,
    tok,
    model_cfg,
    prompt="ROMEO:",
    max_tokens=400,
    temperature=0.55,
    top_k=15,
    seed=0,
)
print(text)


Using /content/Spiking-Neural-Network/checkpoints/llm_large/ckpt_15000_weights.pkl
ROMEO:RLEENTTTTth
TPPAUUTTDEISSTwweeshere poor nee'shhau he mayou lie mayse myourgesto she shouldd
do''twill see your marry shall be gone
To fright the blord for which strain
Thou shalt shall drin thee and her her is hate to stal
With his salf, were they see, and to him for a selly.
Will the stard hath strifl'd to displain
To see him that will all the mutime of the lad
This father shall stand to the bel


## 6. Colab only — download checkpoint

In [ ]:
if IN_COLAB:
    from google.colab import files

    files.download(str(ckpt))
else:
    print("Local run — checkpoint already at", ckpt)
print(str(ckpt))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

/content/Spiking-Neural-Network/checkpoints/llm_large/ckpt_15000_weights.pkl


In [11]:
files.download("/content/Spiking-Neural-Network/checkpoints/llm_large/ckpt_15000_weights.pkl")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!ls /content/Sp

sample_data  Spiking-Neural-Network
